## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gemma3:4b" #  "gpt-4.1-nano" "gemma4:31b-cloud"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0, model=MODEL)

### These LangChain objects implement the method `invoke()`

In [12]:
retriever.invoke("Who is Avery?")

[Document(id='579b0e0e-bc43-4601-a459-d31e07565beb', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  \n\n- **2016**: **Meets Expectations**  \n  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  \n\n- **2017**: **Developing**  \n  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  \n\n- **2018**: **Exceeds Expect

In [21]:
llm.invoke("Who is Avery?")

AIMessage(content='Okay, let\'s break down who "Avery" can be, as it\'s a surprisingly common name and has several notable figures associated with it! Here\'s a breakdown of the most prominent Avery\'s:\n\n**1. Avery Anna (TikTok Star & YouTuber):**\n\n* **This is likely who most people are referring to when they say "Avery" these days.** Avery Anna (born November 18, 2006) is a hugely popular TikTok and YouTube creator. She\'s known for:\n    * **Comedy Sketches & Parodies:** She\'s famous for her hilarious and often absurd short-form videos, frequently featuring her twin sister, Avery.\n    * **Duets & Trends:** She actively participates in TikTok trends and duets.\n    * **Large Following:** She has millions of followers on both platforms.\n    * **Personality:** She\'s known for her energetic, goofy, and relatable personality.\n\n\n**2. Avery Fisher (Actress):**\n\n* Avery Fisher is an American actress. She is best known for her role as "Sarah" in the 2018 horror film *Hereditary*.

## Time to put this together!

In [5]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [6]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [7]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has been leading it ever since! \n\nHere’s a bit more about her:\n\n*   **Date of Birth:** March 15, 1985\n*   **Location:** San Francisco, California\n*   **Current Salary:** $225,000\n\nShe’s known for her innovative leadership and risk management expertise, and has a really impressive career history, starting with her time as a Senior Product Manager at Innovate Insurance Solutions before launching Insurellm.'

## What could possibly come next? 😂

In [1]:
gr.ChatInterface(answer_question).launch()

NameError: name 'gr' is not defined

## Admit it - you thought RAG would be more complicated than that!!